# 01d — The one-year survival label

Loads `mc_matched.parquet` (from `01c_id_matching`), builds one row per business with its one-year label,
and saves it as `labels.parquet` for `01e_ssic_classification`.

**Rules for this notebook**
- The raw files are never changed. Anything cleaned is saved as a new file.
- Nothing is dropped. Problem rows get a flag column instead.
- Data never goes into Git. Clear all outputs before committing.

**The preprocessing notebooks, in order** (each one loads what the one before saved):
1. `01a_id_check`: did the ID change, or only its column name? (stands alone)
2. `01b_all_quarters`: reads the 30 Excel files (2023 Q1 – 2026 Q3) once and saves `mc_all_quarters.parquet`
3. `01c_id_matching`: one ID per business, saves `mc_matched.parquet`
4. `01d_survival_label`: the one-year label, saves `labels.parquet`
5. `01e_ssic_classification`: SSIC from the name, saves `ssic_by_business.csv`, `business_names.csv`, `business_city.csv`
6. `01f_old_data_branches`: old data (2023 Q1 – 2025 Q2), every branch joined to its main (stands alone)

### Cell 1 — Setup
Finds the data. On Colab it downloads the raw files from the team's Google Drive folder (temporary copies,
gone when the session ends). On a laptop it reads the local copy in `DATA_DIR`.

The results that the next notebook loads go in `SAVED_DIR`: the folder **saved** inside the team's shared Drive
folder **Darb DataSets**. Everyone loads the same results, and they are still there in a new Colab session.
On Colab: add a shortcut to Darb DataSets in your My Drive first (Drive → right-click the folder → Organize →
Add shortcut → My Drive). Colab asks once to connect Drive.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    !pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
    from google.colab import drive
    drive.mount('/content/drive')
    TEAM_DIR = Path('/content/drive/MyDrive/Darb DataSets')     # the team's shared folder (a shortcut in My Drive)
    if not TEAM_DIR.exists():
        raise FileNotFoundError('Darb DataSets is not in your My Drive: add a shortcut to it (see the text above), then run this cell again.')
    SAVED_DIR = TEAM_DIR / 'saved'                              # shared with the team, kept between Colab sessions
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', r'C:\Users\Nasser Alyahya\darb-data'))
    SAVED_DIR = DATA_DIR / 'saved'

MC_DIR = DATA_DIR / 'MC'
SAVED_DIR.mkdir(parents=True, exist_ok=True)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR, '| saved results:', SAVED_DIR)

### Load — the matched table from `01c_id_matching`
Only the columns the label needs, to save time and memory.

In [ ]:
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PATH = SAVED_DIR / 'mc_matched.parquet'
if not PATH.exists():
    raise FileNotFoundError(f'{PATH} not found: run 01c_id_matching first.')
# Text is loaded in the same compact format as in 01c_id_matching (pd.read_parquet would use about twice the memory)
TEXT = pd.StringDtype('pyarrow')
mc = pq.read_table(PATH, columns=['entity_id', 'id', 'quarter', 'start_date', 'record_type', 'link_flag', 'recovered_id']).to_pandas(types_mapper={pa.string(): TEXT, pa.large_string(): TEXT}.get)
print(f'Loaded {PATH.name}: {len(mc):,} rows | memory used: {mc.memory_usage(deep=True).sum() / 1e9:.1f} GB')

### Cell 14 — Build the label (one row per business)
**Method**
1. **Unit:** one business = one `entity_id` (from `01c_id_matching` Cell 13), main records only. A business that re-registered
   under a new ID counts once. The duplicated row and rows with no ID are not counted. 2025 Q3 rows whose ID
   was recovered (`01c_id_matching` Cell 13b) are counted.
2. **Cohort:** the quarter of the business's start date (the original business's date if it re-registered).
3. **One-year check:** the quarter 4 quarters later (cohort 2025 Q2 → 2026 Q2).
4. **Label:** `survived_1y` = 1 if the business is in the one-year quarter (under any of its IDs),
   0 if it is not (it disappeared; the files list only active registrations, so the reason is unknown).
5. **2025 Q3 is incomplete: proof of life only.** Being in it means alive; being missing from it means nothing.
   For the 2024 Q3 cohort (checked in 2025 Q3): in 2025 Q3 or in a later complete quarter = 1; not in
   2025 Q2 (before the 1-year mark) = 0; in 2025 Q2 and never again = undecided (it closed around the 1-year mark).
6. **One row per business** in the table `labels`, with `cohort`, `cohort_quality`, `survived_1y` and
   `label_reason` (`present_after_1_year`, `came_back_with_new_id`, `disappeared`, `missing_at_1_year_back_later`,
   `seen_after_1_year`, `disappeared_before_1_year`, `gone_around_1_year`, `cannot_judge`).

`cohort_quality` says how much a cohort can be trusted (`clean`, `start_incomplete`, `start_missing`,
`start_and_check_incomplete`). Every other business stays in the table with `cannot_judge`.
Undecided businesses (empty `survived_1y`) are shown and decided together.

Needs Cell 1 and the Load cell run first.

In [ ]:
import gc
import numpy as np

QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']
TRUSTED = ['2025Q2', '2025Q4', '2026Q1', '2026Q2', '2026Q3']     # complete: can say "this business is gone" (Cell 10)
ALIVE_PROOF = TRUSTED + ['2025Q3']                                 # can say "this business is there"
# 2025 Q3 is incomplete: being in it proves a business is alive, being missing from it proves nothing.

# Cohorts that get a label, and how much we trust them
COHORT_QUALITY = {
    '2025Q2': 'clean',                       # start file and 1-year file both complete
    '2025Q1': 'start_incomplete',            # start file ~85% complete
    '2024Q4': 'start_incomplete',            # start file ~64% complete
    '2025Q3': 'start_missing',               # start file has none of the new businesses
    '2024Q3': 'start_and_check_incomplete',  # start file ~55% complete, 1-year file (2025 Q3) = proof of life only
}

# 1. The unit: one business = one entity_id, main records only.
#    Not counted: the 2nd copy of a duplicated row, and rows with no ID.
#    Rows whose ID was recovered (Cell 13b) are counted, with their recovered ID.
USE_FLAGS = ['same_id', 'linked_new_id', 'ambiguous_not_linked', 'recovered_id']
use = mc.loc[(mc['record_type'] == 'رئيسي') & mc['link_flag'].isin(USE_FLAGS),
             ['entity_id', 'id', 'quarter', 'start_date', 'link_flag']]
if 'recovered_id' in mc.columns:
    use = use.assign(id=use['id'].fillna(mc.loc[use.index, 'recovered_id']))
print(f'Recovered 2025 Q3 rows used (main records): {int((use["link_flag"] == "recovered_id").sum()):,}'
      + ('' if 'recovered_id' in mc.columns else '  (Cell 13b not run)'))
use = use.assign(start_day=use['start_date'].str[:10]).drop(columns='start_date')

# One row per business. Its start date is the earliest start date of its IDs
# (for a re-registered business, that is the original business's date).
labels = use.groupby('entity_id', observed=True).agg(start_day=('start_day', 'min'))
labels['ambiguous_link'] = labels.index.isin(use.loc[use['link_flag'] == 'ambiguous_not_linked', 'entity_id'].unique())

# 2. The cohort: the quarter the business started in. 3. The 1-year check: 4 quarters later.
start_quarter = pd.to_datetime(labels['start_day'], errors='coerce').dt.to_period('Q')
labels['cohort'] = start_quarter.astype(str)
labels['one_year_quarter'] = (start_quarter + 4).astype(str)
labels['cohort_quality'] = labels['cohort'].map(COHORT_QUALITY).fillna('cannot_judge')
eligible = labels['cohort_quality'] != 'cannot_judge'

# Which businesses (entity_id) and which IDs are present in each quarter we can use
needed = sorted(set(labels.loc[eligible, 'one_year_quarter']))
entities_in = {q: pd.Index(use.loc[use['quarter'] == q, 'entity_id'].unique()) for q in ALIVE_PROOF}
ids_in = {q: pd.Index(use.loc[use['quarter'] == q, 'id'].dropna().unique()) for q in needed}

# 4. The label: 1 = still there after 1 year, 0 = disappeared (reason unknown), empty = undecided
survived = pd.Series(pd.NA, index=labels.index, dtype='Int8')
reason = pd.Series('cannot_judge', index=labels.index, dtype=object)
for q in needed:
    rows = (eligible & (labels['one_year_quarter'] == q)).to_numpy()
    businesses = labels.index[rows]
    present = businesses.isin(entities_in[q])             # under any of its IDs
    own_id_present = businesses.isin(ids_in[q])           # under its original ID
    seen_later = np.zeros(len(businesses), dtype=bool)    # in a complete quarter after the check
    for later_q in TRUSTED:
        if later_q > q:
            seen_later |= businesses.isin(entities_in[later_q])
    seen_later &= ~present
    values = np.where(present, 1, 0).astype(float)

    if q in TRUSTED:
        # Complete file: missing = gone (0), unless it is back in a later quarter (undecided)
        missing = np.where(seen_later, 'missing_at_1_year_back_later', 'disappeared')
        values[seen_later] = np.nan
    else:
        # Incomplete file (2025 Q3): missing proves nothing.
        #   seen in a later complete quarter          -> it was alive at 1 year (1)
        #   not in the last complete quarter BEFORE   -> it closed before 1 year (0)
        #   in that quarter, then never again         -> it closed around the 1-year mark (undecided)
        before_q = max(t for t in TRUSTED if t < q)
        in_before = businesses.isin(entities_in[before_q])
        missing = np.where(seen_later, 'seen_after_1_year',
                  np.where(in_before, 'gone_around_1_year', 'disappeared_before_1_year'))
        values[seen_later] = 1
        values[~present & ~seen_later & in_before] = np.nan

    reason[rows] = np.where(present & own_id_present, 'present_after_1_year',
                   np.where(present, 'came_back_with_new_id', missing))
    survived[rows] = pd.array(values, dtype='Int8')

labels['survived_1y'] = survived
labels['label_reason'] = reason.astype('category')
del use, entities_in, ids_in, survived, reason
gc.collect()

# Check: exactly one row per business, no duplicates
print(f'Businesses in the label table: {len(labels):,} | one row each (no duplicates): {labels.index.is_unique}')
print()

# 5. Survival rate per cohort
labelled = labels[eligible]
per_cohort = labelled.groupby('cohort').agg(
    quality=('cohort_quality', 'first'),
    businesses=('survived_1y', 'size'),
    survived=('survived_1y', lambda s: int((s == 1).sum())),
    failed=('survived_1y', lambda s: int((s == 0).sum())),
    undecided=('survived_1y', lambda s: int(s.isna().sum())),
)
per_cohort['survival rate'] = (per_cohort['survived'] / (per_cohort['survived'] + per_cohort['failed'])).map('{:.1%}'.format)
print('Survival per cohort:')
print(per_cohort.to_string())
print()
print('Why each business got its label:')
print(labels['label_reason'].value_counts().to_string())
print()
print(f'Labelled businesses with an unsure ID link (ambiguous_link): {int(labelled["ambiguous_link"].sum()):,}')


### Save — the label table for the next notebook
Saves `labels` (one row per business, index `entity_id`) to `labels.parquet` in `SAVED_DIR`.
`01e_ssic_classification` loads it.

In [ ]:
OUT = SAVED_DIR / 'labels.parquet'
labels.to_parquet(OUT, index=True)
print(f'Saved {len(labels):,} rows to {OUT} ({OUT.stat().st_size / 1e6:,.0f} MB)')